In [1]:
import requests
import json
from dotenv import load_dotenv
import os
import pandas as pd
load_dotenv()
API_KEY = os.getenv("API_KEY")
authors_file = open('authors.txt','r')

In [2]:
import requests

rows = []

for author in authors_file:
    URL = "https://api.openalex.org/authors"

    params = {
        "filter": f"display_name.search:{author}",
        "select": "display_name,works_count,id,works_api_url,last_known_institutions,summary_stats",
        "api_key": API_KEY
    }

    response = requests.get(URL, params=params)
    data = response.json()
    if data.get("results"):
        data = data["results"][0]  # Get the first result
        h_index = data.get("summary_stats", {}).get("h_index")

        # Get first institution country code
        institutions = data.get("last_known_institutions", [])
        country_code = institutions[0].get("country_code") if institutions else None
        rows.append({
                "display_name": data.get("display_name"),
                "works_count": data.get("works_count"),
                "author_id": data.get("id"),
                "works_api_url": data.get("works_api_url"),
                "h_index": h_index,
                "country_code": country_code
            })
df = pd.DataFrame(rows)
df.to_csv("authors_data.csv", index=False)